In [29]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,roc_auc_score

In [30]:
#Model Creation

#load the cleaned_datawset

x_train_data = pd.read_csv("../dataset/x_class_train.csv")
x_test_data = pd.read_csv("../dataset/x_class_test.csv")

y_train_data = pd.read_csv("../dataset/y_class_train.csv")
y_test_data = pd.read_csv("../dataset/y_class_test.csv")
print(x_train_data.columns)
x_train_data = x_train_data.drop(columns=["id"])
x_test_data = x_test_data.drop(columns=["id"])

model = DecisionTreeClassifier( random_state=42)

model.fit(x_train_data, y_train_data)

train_pred = model.predict(x_train_data)
test_pred = model.predict(x_test_data)

train_accuracy = accuracy_score(y_train_data,train_pred)
test_accuracy = accuracy_score(y_test_data, test_pred)

print(f"Taining Accuracy : {train_accuracy*100:.2f}")
print(f"Test Accuracy : {test_accuracy*100:.2f}")

Index(['id', 'age', 'hypertension', 'heart_disease', 'avg_glucose_level',
       'bmi', 'gender_Male', 'gender_Other', 'ever_married_Yes',
       'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban',
       'smoking_status_formerly smoked', 'smoking_status_never smoked',
       'smoking_status_smokes'],
      dtype='object')
Taining Accuracy : 99.93
Test Accuracy : 91.42


In [31]:
#CONTROLLED DECISION TREE
model = DecisionTreeClassifier( random_state=42,max_depth=5, min_samples_split=20)

model.fit(x_train_data, y_train_data)

train_pred = model.predict(x_train_data)
test_pred = model.predict(x_test_data)

train_accuracy = accuracy_score(y_train_data,train_pred)
test_accuracy = accuracy_score(y_test_data, test_pred)

print(f"Taining Accuracy : {train_accuracy*100:.2f}")
print(f"Test Accuracy : {test_accuracy*100:.2f}")

Taining Accuracy : 95.30
Test Accuracy : 93.37


In [32]:
#GINI VS ENTROPY

gini_model = DecisionTreeClassifier( criterion = "gini", max_depth= 5, random_state=42)
gini_model.fit(x_train_data, y_train_data)
gini_pred = gini_model.predict(x_test_data)
gini_acc = accuracy_score(y_test_data,gini_pred)


entropy_model = DecisionTreeClassifier( criterion = "entropy", max_depth= 5, random_state=42)
entropy_model.fit(x_train_data, y_train_data)
entropy_pred = entropy_model.predict(x_test_data)
entropy_acc = accuracy_score(y_test_data,entropy_pred)

print(f"Gini Test Accuracy : {gini_acc*100:.2f}")
print(f"Enthropy Test Accuracy :{entropy_acc*100:.2f}")


Gini Test Accuracy : 93.57
Enthropy Test Accuracy :93.66


In [37]:
#RANDOM FOREST

randomForest_model = RandomForestClassifier ( n_estimators =100, max_depth=10, random_state = 42)

randomForest_model.fit(x_train_data,y_train_data.squeeze())
train_pred =randomForest_model.predict(x_train_data)
test_pred = randomForest_model.predict(x_test_data)

test_prob = randomForest_model.predict_proba(x_test_data)[:, 1]

train_accuracy = accuracy_score(y_train_data, train_pred)
test_accuracy = accuracy_score(y_test_data, test_pred)
auc = roc_auc_score(y_test_data, test_prob)
print(f" Random Forest classifier")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"ROC-AUC           : {auc:.4f}")
print()

#FEATURE IMPRTANCE
importance_df = pd.DataFrame({
 "Features":x_train_data.columns,
    "Importance":randomForest_model.feature_importances_
}).sort_values( by = "Importance",ascending=False)

print(f"Top 5 important features")
print(importance_df.head())

 Random Forest classifier
Training Accuracy : 0.9725
Test Accuracy     : 0.9376
ROC-AUC           : 0.7891

Top 5 important features
                Features  Importance
0                    age    0.273889
3      avg_glucose_level    0.253989
4                    bmi    0.214875
12  Residence_type_Urban    0.033514
5            gender_Male    0.032474


In [39]:
from sklearn.ensemble import GradientBoostingClassifier

gbc_model = GradientBoostingClassifier( n_estimators = 100, learning_rate=0.1,
                                        max_depth=3, random_state=42)

gbc_model.fit(x_train_data, y_train_data.squeeze())

train_pred = gbc_model.predict(x_train_data)
test_pred = gbc_model.predict(x_test_data)

test_prob = gbc_model.predict_proba(x_test_data)[:,1]

gbc_train_accuracy = accuracy_score(y_train_data, train_pred)
gbc_test_accuracy = accuracy_score(y_test_data, test_pred)
gbc_auc_score = roc_auc_score(y_test_data,test_prob)

print(f"Gradient Boosting Classifier")
print(f"Training Accuracy : {gbc_train_accuracy:.2f}")
print(f"Test Accuracy : {gbc_test_accuracy:.2f}")
print(f"ROC-AUC score : {gbc_auc_score:.2f}")

Gradient Boosting Classifier
Training Accuracy : 0.96
Test Accuracy : 0.93
ROC-AUC score : 0.81


In [52]:
#Feature ABlation Study
# 5 lowest feature_importances

lowest_features = importance_df.sort_values(by="Importance",ascending=True)
print(f" {lowest_features.head()}")
print()
features_to_remove = lowest_features["Features"].head().tolist()
print(f" Features to Remove " )
print(features_to_remove)
print()
print("RandomForest classifer with reduced features ")
x_train_new = x_train_data.drop(columns=features_to_remove)
x_test_new = x_test_data.drop(columns=features_to_remove)

rf_model_new = RandomForestClassifier (n_estimators =100, max_depth=10, random_state=42)
rf_model_new.fit(x_train_new, y_train_data.squeeze())

prob_new = rf_model_new.predict_proba(x_test_new)[:,1]
auc_new = roc_auc_score(y_test_data,prob_new)
print()
print(f" Model ROC- AUC : {auc_new:.2f}")

                        Features  Importance
6                  gender_Other    0.000000
8        work_type_Never_worked    0.000014
11           work_type_children    0.001860
15        smoking_status_smokes    0.017942
14  smoking_status_never smoked    0.021842

 Features to Remove 
['gender_Other', 'work_type_Never_worked', 'work_type_children', 'smoking_status_smokes', 'smoking_status_never smoked']

RandomForest classifer with reduced features 

 Model ROC- AUC : 0.78


In [54]:
#Comparison of two randomForest classifier model
print("Comparison of two randomForest classifier model")
comparison = pd.DataFrame({
    "Model":["Full Model","Reduced Model"],
    "ROC-AUC score":[auc,auc_new]
})

print()
print(comparison)

Comparison of two randomForest classifier model

           Model  ROC-AUC score
0     Full Model       0.789066
1  Reduced Model       0.783607


In [58]:
# CROSS_VALIDATION COMPARISON
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


print("Cross Validation Comparison")
path = "../dataset/cleaned_data.csv"
data_df = pd.read_csv(path)

x_claasification_features = data_df.drop(columns=["stroke","id"])
y_classification_target = data_df["stroke"]

categorical_cols = ["gender","ever_married","work_type","Residence_type","smoking_status"]

encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False
)

encoder_data = encoder.fit_transform(data_df[categorical_cols])
print("Encoder Shape:", encoder_data.shape)

feature_names = encoder.get_feature_names_out(categorical_cols)

print("Number of feature names:", len(feature_names))
print(feature_names)

#enncoded data into DataFrame
encoded_data_df = pd.DataFrame( encoder_data, columns = feature_names, index=data_df.index)

x_class_train = x_claasification_features.drop(columns = categorical_cols)
x_class_train = pd.concat([x_class_train,encoded_data_df],axis=1)

cv = StratifiedKFold( n_splits=5,shuffle=True,random_state=42)


models = {
    "Logistic Regression" : LogisticRegression(class_weight="balanced",max_iter=1000,random_state=42),
    "Decision Tree" : DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest" : RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42),
    "Gradient Boosting":GradientBoostingClassifier(n_estimators =100, learning_rate=0.1,max_depth=3,random_state=42)
}


results =[]

for name, model in models.items():
    scores = cross_val_score(model,x_class_train,y_classification_target,
                             cv=cv,scoring="roc_auc")

    results.append({
        "Model":name,
        "Mean ROC-AUC score ":scores.mean(),
        "Standard deviation":scores.std()
    })

cv_results = pd.DataFrame(results)
print(f" Cross-Validation Results :")
print(cv_results)

Cross Validation Comparison
Encoder Shape: (5130, 11)
Number of feature names: 11
['gender_Male' 'gender_Other' 'ever_married_Yes' 'work_type_Never_worked'
 'work_type_Private' 'work_type_Self-employed' 'work_type_children'
 'Residence_type_Urban' 'smoking_status_formerly smoked'
 'smoking_status_never smoked' 'smoking_status_smokes']
 Cross-Validation Results :
                 Model  Mean ROC-AUC score   Standard deviation
0  Logistic Regression             0.815782            0.009800
1        Decision Tree             0.794503            0.014910
2        Random Forest             0.827172            0.028313
3    Gradient Boosting             0.837435            0.026170


In [64]:
#Hyperparamter Tuning
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import train_test_split


param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

pipeline = make_pipeline(SimpleImputer(strategy = 'median'),
                        StandardScaler(),RandomForestClassifier(random_state=42))


x_train, x_test, y_train, y_test = train_test_split(
    x_class_train,
    y_classification_target,
    test_size=0.2,
    stratify=y_classification_target,
    random_state=42
)
                         
cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    cv = cv,
    scoring = "roc_auc",
    n_jobs =1
)

grid_search.fit(x_train,y_train)

print(f"Best Performance : {grid_search.best_params_}")
print()
print(f"Best CV ROC_AUC : {grid_search.best_score_:.2f}")

Best Performance : {'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 200}

Best CV ROC_AUC : 0.85


In [65]:
#Manual learning curve

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

best_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        random_state=42
    )
)

In [66]:

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for f in fractions:

    n = int(f * len(x_train))

    X_subset = x_train.iloc[:n]
    y_subset = y_train.iloc[:n]

    best_pipeline.fit(X_subset, y_subset)
    
    train_prob = best_pipeline.predict_proba(X_subset)[:,1]
    test_prob = best_pipeline.predict_proba(x_test)[:,1]

    train_auc = roc_auc_score(y_subset, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    results.append({
        "Training Fraction": f,
        "Training AUC": train_auc,
        "Test AUC": test_auc
    })

learning_curve = pd.DataFrame(results)

print(f"{learning_curve}")

   Training Fraction  Training AUC  Test AUC
0                0.2      0.980260  0.809081
1                0.4      0.982610  0.818544
2                0.6      0.984624  0.823101
3                0.8      0.986079  0.825641
4                1.0      0.985826  0.827076


In [67]:
import joblib

best_pipeline = grid_search.best_estimator_

joblib.dump(best_pipeline, "best_model.pkl")

print("Model saved successfully!")

loaded_model = joblib.load("best_model.pkl")

print("Model loaded successfully!")
sample_data = x_test.iloc[:2]

print(sample_data)

predictions = loaded_model.predict(sample_data)

print("Predictions:")
print(predictions)

Model saved successfully!
Model loaded successfully!
       age  hypertension  heart_disease  avg_glucose_level   bmi  gender_Male  \
69    76.0             0              0             104.47  20.3          1.0   
4515  43.0             0              0              89.73  23.5          0.0   

      gender_Other  ever_married_Yes  work_type_Never_worked  \
69             0.0               1.0                     0.0   
4515           0.0               1.0                     0.0   

      work_type_Private  work_type_Self-employed  work_type_children  \
69                  1.0                      0.0                 0.0   
4515                0.0                      1.0                 0.0   

      Residence_type_Urban  smoking_status_formerly smoked  \
69                     1.0                             0.0   
4515                   1.0                             1.0   

      smoking_status_never smoked  smoking_status_smokes  
69                            0.0              